# 🌳 Trabalho Prático — Árvores de Decisão
**Cadeira de Exploração de Dados · ISCIM**

> Pipeline completa: EDA → Pré-processamento → Treino → Avaliação → Visualização

In [ ]:
# ── Célula 1 — Importações ─────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import datasets
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('✅ Bibliotecas carregadas com sucesso!')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# ⚙️  CONFIGURAÇÃO DO DATASET — O DOCENTE ALTERA APENAS ESTA SECÇÃO
# ═══════════════════════════════════════════════════════════════════
#
# Opções disponíveis:
#   'iris'          → dataset Iris (sklearn)
#   'wine'          → dataset Wine (sklearn)
#   'breast_cancer' → dataset Breast Cancer (sklearn)
#   'csv'           → ficheiro CSV externo (definir CAMINHO_CSV)
#
DATASET = 'iris'

# Se DATASET == 'csv', definir o caminho e coluna alvo:
CAMINHO_CSV  = 'dados.csv'
COLUNA_ALVO  = 'target'   # nome da coluna a prever

# Parâmetros da Árvore (podem ser ajustados)
MAX_DEPTH    = 4          # profundidade máxima
TEST_SIZE    = 0.25       # proporção do conjunto de teste
RANDOM_STATE = 42         # semente para reprodutibilidade
CRITERION    = 'gini'     # 'gini' ou 'entropy'
# ═══════════════════════════════════════════════════════════════════

In [ ]:
# ── Célula 3 — Carregamento do Dataset ────────────────────────────
def carregar_dataset(nome):
    """Carrega o dataset escolhido e devolve (X, y, nomes_features, nomes_classes)."""
    if nome == 'iris':
        ds = datasets.load_iris()
    elif nome == 'wine':
        ds = datasets.load_wine()
    elif nome == 'breast_cancer':
        ds = datasets.load_breast_cancer()
    elif nome == 'csv':
        df = pd.read_csv(CAMINHO_CSV)
        X_raw = df.drop(columns=[COLUNA_ALVO])
        y_raw = df[COLUNA_ALVO]
        for col in X_raw.select_dtypes(include='object').columns:
            X_raw[col] = LabelEncoder().fit_transform(X_raw[col])
        le = LabelEncoder()
        y_enc = le.fit_transform(y_raw)
        return (
            X_raw.fillna(X_raw.median(numeric_only=True)).values,
            y_enc,
            X_raw.columns.tolist(),
            le.classes_.tolist()
        )
    else:
        raise ValueError(f'Dataset desconhecido: {nome}')

    return (
        ds.data,
        ds.target,
        ds.feature_names.tolist() if hasattr(ds.feature_names, 'tolist') else list(ds.feature_names),
        ds.target_names.tolist()
    )


X, y, FEATURES, CLASSES = carregar_dataset(DATASET)
print(f'Dataset   : {DATASET.upper()}')
print(f'Amostras  : {X.shape[0]}  |  Features: {X.shape[1]}  |  Classes: {len(CLASSES)}')
print(f'Classes   : {CLASSES}')

In [ ]:
# ── Célula 4 — Análise Exploratória (EDA) ─────────────────────────
df = pd.DataFrame(X, columns=FEATURES)
df['classe'] = [CLASSES[i] for i in y]

print('=' * 55)
print('PRIMEIRAS LINHAS')
display(df.head(5))

print('\nESTATÍSTICAS DESCRITIVAS')
display(df.describe().round(2))

print('\nVALORES EM FALTA')
display(df.isnull().sum())

# Gráficos EDA
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

contagem = df['classe'].value_counts()
axes[0].bar(contagem.index, contagem.values,
            color=sns.color_palette('muted', len(contagem)))
axes[0].set_title('Distribuição das Classes')
axes[0].set_xlabel('Classe')
axes[0].set_ylabel('Contagem')

corr = pd.DataFrame(X, columns=FEATURES).corr()
sns.heatmap(corr, ax=axes[1], annot=True, fmt='.1f',
            cmap='coolwarm', linewidths=.5, square=True)
axes[1].set_title('Correlação entre Features')

plt.tight_layout()
plt.show()

In [ ]:
# ── Célula 5 — Divisão dos Dados e Treino ─────────────────────────
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)
print(f'Treino: {len(X_treino)} amostras  |  Teste: {len(X_teste)} amostras')

modelo = DecisionTreeClassifier(
    max_depth=MAX_DEPTH,
    criterion=CRITERION,
    random_state=RANDOM_STATE
)
modelo.fit(X_treino, y_treino)
print('✅ Modelo treinado!')

scores_cv = cross_val_score(modelo, X, y, cv=5, scoring='accuracy')
print(f'Accuracy (CV 5-fold): {scores_cv.mean():.3f} ± {scores_cv.std():.3f}')

In [ ]:
# ── Célula 6 — Avaliação do Modelo ────────────────────────────────
y_pred = modelo.predict(X_teste)

print(f'Accuracy no Teste : {accuracy_score(y_teste, y_pred):.4f}')
print('\nRelatório de Classificação:')
print(classification_report(y_teste, y_pred, target_names=CLASSES))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_teste, y_pred,
    display_labels=CLASSES,
    cmap='Blues',
    ax=ax
)
ax.set_title('Matriz de Confusão')
plt.tight_layout()
plt.show()

In [ ]:
# ── Célula 7 — Visualização da Árvore ────────────────────────────
fig, ax = plt.subplots(figsize=(18, 8))
plot_tree(
    modelo,
    feature_names=FEATURES,
    class_names=CLASSES,
    filled=True,
    rounded=True,
    fontsize=9,
    ax=ax
)
ax.set_title(f'Árvore de Decisão — {DATASET.upper()} (profundidade={MAX_DEPTH})', fontsize=13)
plt.tight_layout()
plt.savefig('arvore_decisao.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n── Regras de Decisão ──\n')
print(export_text(modelo, feature_names=FEATURES))

In [ ]:
# ── Célula 8 — Importância das Features ──────────────────────────
importancias = pd.Series(
    modelo.feature_importances_, index=FEATURES
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
importancias.plot(kind='barh', ax=ax,
                  color=sns.color_palette('muted', len(importancias)))
ax.set_title('Importância das Features (Gini)')
ax.set_xlabel('Importância relativa')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(importancias.to_string())

In [ ]:
# ── Célula 9 (Facultativa) — Análise de Overfitting ───────────────
profundidades = range(1, 15)
acc_treino, acc_teste = [], []

for d in profundidades:
    m = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE)
    m.fit(X_treino, y_treino)
    acc_treino.append(accuracy_score(y_treino, m.predict(X_treino)))
    acc_teste.append(accuracy_score(y_teste, m.predict(X_teste)))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(profundidades, acc_treino, marker='o', label='Treino')
ax.plot(profundidades, acc_teste,  marker='s', label='Teste')
ax.axvline(MAX_DEPTH, color='red', ls='--', lw=1, label=f'max_depth={MAX_DEPTH}')
ax.set_xlabel('Profundidade da Árvore')
ax.set_ylabel('Accuracy')
ax.set_title('Análise de Overfitting por Profundidade')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Célula 10 (Facultativa) — GridSearchCV ─────────────────────────
param_grid = {
    'max_depth'        : [2, 3, 4, 5, 6, 8, None],
    'criterion'        : ['gini', 'entropy'],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf' : [1, 2, 4],
}

grid = GridSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid.fit(X_treino, y_treino)

print('Melhores hiperparâmetros:', grid.best_params_)
print(f'Melhor Accuracy (CV) : {grid.best_score_:.4f}')

melhor = grid.best_estimator_
y_pred_otimizado = melhor.predict(X_teste)
print(f'Accuracy no Teste (otimizado): {accuracy_score(y_teste, y_pred_otimizado):.4f}')